In [1]:
import os
import torch
import random
import torch.nn as nn
import torch.backends.cudnn as cudnn
from models import build_model
import numpy as np
from PIL import Image

/opt/anaconda3/envs/LaVIT-main/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


Please 'pip install apex'
Please 'pip install apex'
Please 'pip install apex'
Please 'pip install apex'


In [2]:
# The local directory to save LaVIT checkpoint, set to yours
model_path = "/data/phd/wsun/checkpoint_LaVIT/VideoLaVIT-v1/language_model_sft"   # 例：/home/jinyang06/models/VideoLaVIT-v1/language_model_sft
model_dtype='bf16'

max_video_clips = 16
device_id = 0
torch.cuda.set_device(device_id)
device = torch.device('cuda')

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# For Multi-Modal Understanding
runner = build_model(model_path=model_path, model_dtype=model_dtype, understanding=True,
        device_id=device_id, use_xformers=True, max_video_clips=max_video_clips,)
print("Building Model Finsished")

Loading Video LaVIT Model Weight from /data/phd/wsun/checkpoint_LaVIT/VideoLaVIT-v1/language_model_sft, model precision: bf16
Not used {}


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at /data/phd/wsun/checkpoint_LaVIT/VideoLaVIT-v1/language_model_sft were not used when initializing VideoLaVITLlamaForCausalLM: ['model.motion_tokenizer.quantize.embedding.initted', 'model.motion_tokenizer.quantize.embedding.embed_avg', 'model.motion_tokenizer.quantize.cluster_size', 'model.motion_tokenizer.quantize.embedding.cluster_size']
- This IS expected if you are initializing VideoLaVITLlamaForCausalLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing VideoLaVITLlamaForCausalLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


The Visual Vocab Size is 16384
The llama tokenizer vocab size is 32000
The maximal clip number is 16
Building Model Finsished


### Image Understanding

In [5]:
image_path = 'demo/03-Confusing-Pictures.jpg'
prompt = "What is unusual about this image?"

output = runner({"image": image_path, "text_input": prompt}, length_penalty=1, temperature=1.0, \
                use_nucleus_sampling=False, num_beams=1, truct_vqa=False, max_length=512,)[0]
print(output)


/opt/anaconda3/envs/LaVIT-main/lib/python3.10/site-packages/transformers/generation/utils.py:1417: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(


The unusual aspect of this image is that a man is ironing clothes on an ironing board placed on the back of a yellow SUV. This is not a typical scene, as ironing is usually done indoors, in a home or laundry room, and not on a moving vehicle. The man's choice to iron clothes in this manner is unconventional and unexpected, as it poses potential risks and safety concerns.


### Video Understanding

In [8]:
video_path = '/data/phd/wsun/datasets/test/01April_2010_Thursday_heute-6694.mp4'
prompt = "Please describe the video and provide a structured text description of her actions sequences. Here is the template: action_desc: [ Both hands bent, knuckles facing, rotate upward, Index finger pointing toward the other person, Index finger extended up, curls toward self in beckoning motion ], Please output the described information following the template"

output = runner({"video": video_path, "text_input": prompt}, length_penalty=1, \
        use_nucleus_sampling=False, num_beams=1, max_length=512, temperature=1.0)[0]
print(output)

The woman in the video is seen making a gesture with her hands. She starts by bending her hands and pointing her index finger upward. She then curls her fingers inward and extends her index finger upward. The gesture seems to be a beckoning motion, as if she is inviting someone to come closer. The woman's body language suggests that she is trying to communicate something to the other person. It is unclear what she is saying, but her gesture is clear and intentional. Overall, the video captures a brief moment of nonverbal communication between two people.
